In [1]:
import requests
import mysql.connector
import pandas as pd
from bs4 import BeautifulSoup
from sqlConfig import loginConfig
import tableSetUpMethods as tsum
import time
from concurrent.futures import ThreadPoolExecutor

In [2]:
cnx = mysql.connector.connect(
    user=loginConfig['user'],
    password=loginConfig['password'],
    host=loginConfig['host'],
    database=loginConfig['database']
)
cursor = cnx.cursor()

In [ ]:
#cursor.close()
#cnx.close()

I know how I can add fighterIDs to the fightStat table. Unfortunately the fightStats build table will have to be edited again. When we are grabbing the fight details we should also grab the reference links. Once we have the reference links check against fighterHyperlinks and if we have a corresponding row for that fighter grab the fighterID

This is assuming fighterHyperlinks is up to date, so we'll have to rerun tableSetup.py since its a little outdated. 

In [41]:
fightLink = 'http://ufcstats.com/fight-details/74f5cbecac011abb'
response = requests.get(fightLink)
soup = BeautifulSoup(response.content, 'html.parser')

tables = soup.find_all('table')
if tables:
    table = tables[0] 
    hyperLinks = table.find_all('a', class_='b-link b-link_style_black')
links = [link['href'] for link in hyperLinks]
links

['http://ufcstats.com/fighter-details/792be9a24df82ed6',
 'http://ufcstats.com/fighter-details/6e15f63b6c2e2c15']

In [42]:
def add_www_to_links(links):
    updated_links = []
    for link in links:
        updated_link = link.replace('http://', 'http://www.')
        updated_links.append(updated_link)
    return updated_links

# Call the function
links = add_www_to_links(links)


In [43]:
links

['http://www.ufcstats.com/fighter-details/792be9a24df82ed6',
 'http://www.ufcstats.com/fighter-details/6e15f63b6c2e2c15']

In [44]:
query = f"""
select fighterID
FROM fighterHyperlinks
where hyperlink = '{links[0]}'
"""
cursor.execute(query)
tmp = cursor.fetchall()
fighterID = tmp[0][0]

In [46]:
for link in links:
    query = f"""
    select fighterID
    FROM fighterHyperlinks
    where hyperlink = '{link}'
    """
    cursor.execute(query)
    tmp = cursor.fetchall()
    fighterID = tmp[0][0]
    print(fighterID)

2509
3186
